# 02 · The recipe: every training setting and the code that sets it

**The question.** Which training settings does the recipe use, why each one, and what does the code add to stock NeuralForecast?
Notebooks 03 to 05 change these settings one at a time and compare against this recipe. Nothing is trained here.

In [1]:
import sys, os
try: import m5knobs  # noqa: F401  (installed with pip install -e .)
except ImportError: sys.path.insert(0, os.path.abspath(".."))  # run from notebooks/
from m5knobs import data, model, paths
from IPython.display import Markdown, display

## The settings and why

- **Loss: Tweedie** (`DistributionLoss("Tweedie", rho=...)`). With rho between 1 and 2 the Tweedie distribution puts real
  probability on exactly zero and has a long right tail, which matches weeks that sell nothing next to occasional big weeks.
- **Scaling: RevIN** (`scaler_type`). Each input window is normalized by its own mean and standard deviation, plus a learnable
  scale and shift, and the output is mapped back, so slow and fast sellers can share one network.
- **Input window** (`input_size`): a full year of weekly history, so the network can see last year's same weeks.
- **Optimizer: NAdam with decoupled weight decay.** Not tested in this repo.
- **Learning rate: one-cycle** (`OneCycleLR`). The rate warms up to its peak over the first 30% of steps, then anneals on a cosine.
  `lr_max` is the peak.
- **Training length and early stopping**: at most `max_steps` optimizer steps; the model is scored on the held-out last
  `val_size` weeks every `val_check_steps` steps and stops after `early_stop_patience_steps` checks without improvement.
  `min_steps` blocks stopping before the learning-rate peak.
- **Best weights restored** (`restore_best`): at the end, the weights from the best validation check are loaded back.
- **Gradient clipping: AutoClip** (`clip_percentile`): each step, the gradient norm is capped at that percentile of the norms seen
  so far in the fit.
- **Short-history filter** (`training_data_availability_threshold`): a training window needs at least this share of real
  (non-padded) input weeks, which keeps windows that are mostly padding out of training. Short series are left-padded
  (`start_padding_enabled=True`). Relaxing the filter is one of the stressors in notebook 04.
- **Batches**: `batch_size` series per step, `windows_batch_size` windows sampled from them, and `drop_last_loader=True`,
  which drops the last partial training batch so every step sees a full batch. None of these is tested in this repo.

The values, read from the code:

In [2]:
import inspect, math, pandas as pd, torch
r = recipe = {**{k: p.default for k, p in inspect.signature(model.Knobs).parameters.items()}, **data.RECIPE}
recipe["min_steps"] = int(0.3 * recipe["max_steps"])  # what Knobs uses when min_steps=None
skip = {"arch", "h", "seed", "accelerator", "freq", "huber_delta"}  # set per run, or only used by the Huber arms
display(pd.DataFrame({"recipe value": {k: str(v) for k, v in recipe.items() if k not in skip}}))

,recipe value
input_size,52
loss,tweedie
clip_percentile,10.0
restore_best,True
lr_max,0.001
max_steps,5000
val_check_steps,50
early_stop_patience_steps,20
min_steps,1500
scaler_type,revin


The exact NeuralForecast call, from `m5knobs/model.py` (every argument is stock NeuralForecast or Lightning except the loss
and the two callbacks, which come from `m5knobs.model` and are described below):

In [3]:
print(inspect.getsource(model.Knobs.__init__))

    def __init__(self, arch="NHITS", h=13, input_size=52, loss="tweedie", seed=1, accelerator="mps", freq="W-SAT",
                 clip_percentile=10.0, restore_best=True, lr_max=1e-3, max_steps=5000, val_check_steps=50,
                 early_stop_patience_steps=20, min_steps=None, scaler_type="revin", huber_delta=5.0, tweedie_rho=1.5,
                 batch_size=128, windows_batch_size=1024, val_size=13, availability_threshold=(0.05, 0.0)):
        self.freq, self.val_size, self.max_steps, self.fold_stats = freq, val_size, max_steps, []
        self._gn, self._lr = AutoClip(clip_percentile), LRCapture()
        callbacks = [self._gn, self._lr] + ([BestWeights()] if restore_best else [])
        sch = torch.optim.lr_scheduler
        model = getattr(_nfm, arch)(
            h=h, input_size=input_size, loss=make_loss(loss, huber_delta, tweedie_rho), scaler_type=scaler_type,
            max_steps=max_steps, val_check_steps=val_check_steps, early_stop_patience_steps=early_stop_patience_

## What `m5knobs/model.py` adds to stock NeuralForecast

- **`AutoClip`** (Seetharaman et al., 2020): a Lightning callback that records the gradient norm every step and clips it to a
  running percentile of past norms.
- **`BestWeights`**: Lightning's early stopping only stops training. This callback loads the best validation check's weights back.
- **`tweedie_zero_guard`**: NeuralForecast's own Tweedie output link with one change, the input window's mean is floored at 0.01
  before the log. Checked below.
- **`KeepValBatchDataModule`**: with `drop_last_loader=True`, stock NeuralForecast also drops the last partial batch of series
  from every validation check. Importing `m5knobs.model` replaces NeuralForecast's data-module class for the whole Python
  process, so validation keeps every series.
- **`pad_short`**: series with no more rows than `val_size` are left-padded with zeros so the validation split exists.

In [4]:
import neuralforecast.common._base_model as nf_base
n, b, patched = data.N_SERIES, recipe["batch_size"], nf_base.TimeSeriesDataModule is model.KeepValBatchDataModule
display(Markdown(f"NeuralForecast now builds `KeepValBatchDataModule`: {patched}. With {n:,} series and "
                 f"`batch_size={b}`, stock validation would skip {n % b} series in every check."))

NeuralForecast now builds `KeepValBatchDataModule`: True. With 30,490 series and `batch_size=128`, stock validation would skip 26 series in every check.

### The Tweedie zero-window guard

The stock link computes `log_mu = softplus(raw) + log(window mean)`, clamped to [1e-9, 37]. An input window of all zeros has mean
0, so the link takes `log(0)`. The clamp hides the `-inf` in the forward pass, but the backward pass multiplies a zero gradient by
`1/0`, which gives NaN. The cell below builds that case with two tiny tensors: one window with mean 0, one with mean 2, and one
learnable weight upstream of the window mean (standing in for RevIN's learnable parameters).

In [5]:
from neuralforecast.losses.pytorch import tweedie_scale_decouple as stock
def grad_into_weight(link):
    w = torch.ones(1, requires_grad=True)
    window_mean = torch.tensor([[0.0, 2.0]]) * w
    log_mu, _ = link((torch.zeros(1, 2, requires_grad=True), 1.5), window_mean, torch.ones(1, 2))
    log_mu.sum().backward()
    return w.grad.item()
g_stock, g_guard = grad_into_weight(stock), grad_into_weight(model.tweedie_zero_guard)

In [6]:
raw, loc, one = torch.tensor([[-3.0, 0.0, 2.0, 40.0]]), torch.tensor([[0.5, 2.0, 7.0, 3.0]]), torch.ones(1, 4)
same = torch.equal(stock((raw, 1.5), loc, one)[0], model.tweedie_zero_guard((raw, 1.5), loc, one)[0])
floor = torch.exp(stock((torch.tensor([[-30.0]]), 1.5), torch.tensor([[0.5]]), torch.ones(1, 1))[0]).item()
display(Markdown(
    f"Gradient into the upstream weight: stock link **{g_stock:.3f}**, with the guard **{g_guard:.3f}**. "
    f"For window means of at least 0.01 the two links return {'identical' if same else 'different'} values. "
    f"The stock link also has a floor that the guard keeps: with a window mean of 0.5 and a very negative raw output, "
    f"the smallest mean it can produce is {floor:.2f} units per week. "
    f"The Tweedie mean cannot go below that floor, nor below the input window's mean. Both are in sales units: the link "
    f"adds the log of the window mean in units, so the floor applies to the forecast itself, not to RevIN's scaled values. "
    f"Notebook 05 shows where forecasts on slow sellers land."))

Gradient into the upstream weight: stock link **nan**, with the guard **1.000**. For window means of at least 0.01 the two links return identical values. The stock link also has a floor that the guard keeps: with a window mean of 0.5 and a very negative raw output, the smallest mean it can produce is 1.00 units per week. The Tweedie mean cannot go below that floor, nor below the input window's mean. Both are in sales units: the link adds the log of the window mean in units, so the floor applies to the forecast itself, not to RevIN's scaled values. Notebook 05 shows where forecasts on slow sellers land.

A NaN gradient reaches every weight it flows through, and the next optimizer update writes NaN into them. One all-zero input
window in a batch is enough. How often such windows occur in M5:

In [7]:
if os.path.exists(paths.M5_WEEKLY):
    p = data.load_panel(); p = p[p.ds <= data.CUTOFFS[0]]
    roll = p.groupby("unique_id")["y"].rolling(r["input_size"]).sum().dropna()
    share, k = float((roll == 0).mean()), r["windows_batch_size"]; prob = 1 - (1 - share) ** k
    display(Markdown(f"Before the first origin, {share:.1%} of the {len(roll):,} full {r['input_size']}-week "
                     f"input windows have no sales. A batch of {k:,} windows drawn uniformly at random would "
                     f"hold at least one with probability {'over 99.9%' if prob > 0.999 else f'{prob:.1%}'}."))
else:
    print("data/m5_weekly.parquet not found: run `python3 scripts/prepare_data.py` to count all-zero input windows.")

Before the first origin, 1.1% of the 4,133,579 full 52-week input windows have no sales. A batch of 1,024 windows drawn uniformly at random would hold at least one with probability over 99.9%.

## What to set

The model arguments and the training arguments, generated from the recipe values above:

In [8]:
display(Markdown(f"""```python
NHITS(h={data.H}, input_size={r['input_size']}, scaler_type="{r['scaler_type']}",
      loss=make_loss("tweedie", tweedie_rho={r['tweedie_rho']}),     # m5knobs.model: stock Tweedie + zero-window guard
      batch_size={r['batch_size']}, windows_batch_size={r['windows_batch_size']}, start_padding_enabled=True,
      training_data_availability_threshold={list(r['availability_threshold'])}, drop_last_loader=True,
```"""))

```python
NHITS(h=13, input_size=52, scaler_type="revin",
      loss=make_loss("tweedie", tweedie_rho=1.5),     # m5knobs.model: stock Tweedie + zero-window guard
      batch_size=128, windows_batch_size=1024, start_padding_enabled=True,
      training_data_availability_threshold=[0.05, 0.0], drop_last_loader=True,
```

In [9]:
display(Markdown(f"""```python
      max_steps={r['max_steps']}, min_steps={r['min_steps']}, val_check_steps={r['val_check_steps']},
      early_stop_patience_steps={r['early_stop_patience_steps']},
      optimizer=torch.optim.NAdam, optimizer_kwargs={{"weight_decay": 0.01, "decoupled_weight_decay": True}},
      lr_scheduler=OneCycleLR, lr_scheduler_kwargs=dict(max_lr={r['lr_max']}, total_steps={r['max_steps'] + 1}, ...),
      callbacks=[AutoClip({r['clip_percentile']}), BestWeights()])      # then nf.fit(df, val_size={r['val_size']})
```"""))

```python
      max_steps=5000, min_steps=1500, val_check_steps=50,
      early_stop_patience_steps=20,
      optimizer=torch.optim.NAdam, optimizer_kwargs={"weight_decay": 0.01, "decoupled_weight_decay": True},
      lr_scheduler=OneCycleLR, lr_scheduler_kwargs=dict(max_lr=0.001, total_steps=5001, ...),
      callbacks=[AutoClip(10.0), BestWeights()])      # then nf.fit(df, val_size=13)
```

`m5knobs.model.Knobs` builds this call; the full call, with every scheduler argument, is printed above. Next: [03 · One setting at a time](03_one_setting_at_a_time.ipynb).